In [1]:
import pandas as pd
import json


In [2]:
# Load maple_gpt4.jsonl
gpt4_data = []
with open('maple_gpt4.jsonl', 'r') as f:
    for line in f:
        gpt4_data.append(json.loads(line))

df_gpt4 = pd.DataFrame(gpt4_data)
print(f"GPT-4 DataFrame shape: {df_gpt4.shape}")
df_gpt4.head()


GPT-4 DataFrame shape: (26, 9)


,class_name,feature_name,model,llm_interactions,success,max_retries_reached,final_status,interactions,code_changes
0,MAPLE_RECURSIVE_ABSOLUTE_1,abs,gpt-4o-mini,1,True,False,Verification passed after 1 LLM interaction(s),"[{'interaction_number': 1, 'error_message': ' ...","[{'change_number': 1, 'before_code': ' if x..."
1,MAPLE_RECURSIVE_ABSOLUTE_2,abs,gpt-4o-mini,1,True,False,Verification passed after 1 LLM interaction(s),"[{'interaction_number': 1, 'error_message': ' ...","[{'change_number': 1, 'before_code': ' if x..."
2,MAPLE_RECURSIVE_CONSEQ_1,sum,gpt-4o-mini,2,True,False,Verification passed after 2 LLM interaction(s),"[{'interaction_number': 1, 'error_message': ' ...","[{'change_number': 1, 'before_code': ' if n..."
3,MAPLE_RECURSIVE_CONSEQ_2,sum,gpt-4o-mini,6,True,False,Verification passed after 6 LLM interaction(s),"[{'interaction_number': 1, 'error_message': ' ...","[{'change_number': 1, 'before_code': ' if n..."
4,MAPLE_RECURSIVE_CONSEQ_3,sum,gpt-4o-mini,6,True,False,Verification passed after 6 LLM interaction(s),"[{'interaction_number': 1, 'error_message': ' ...","[{'change_number': 1, 'before_code': ' if n..."


In [3]:
# Load maple_sonnet.jsonl
sonnet_data = []
with open('maple_sonnet.jsonl', 'r') as f:
    for line in f:
        sonnet_data.append(json.loads(line))

df_sonnet = pd.DataFrame(sonnet_data)
print(f"Sonnet DataFrame shape: {df_sonnet.shape}")
df_sonnet.head()


Sonnet DataFrame shape: (26, 9)


,class_name,feature_name,model,llm_interactions,success,max_retries_reached,final_status,interactions,code_changes
0,MAPLE_RECURSIVE_ABSOLUTE_1,abs,claude-sonnet-4-0,1,True,False,Verification passed after 1 LLM interaction(s),"[{'interaction_number': 1, 'error_message': ' ...","[{'change_number': 1, 'before_code': ' if x..."
1,MAPLE_RECURSIVE_ABSOLUTE_2,abs,claude-sonnet-4-0,1,True,False,Verification passed after 1 LLM interaction(s),"[{'interaction_number': 1, 'error_message': ' ...","[{'change_number': 1, 'before_code': ' if x..."
2,MAPLE_RECURSIVE_CONSEQ_1,sum,claude-sonnet-4-0,1,True,False,Verification passed after 1 LLM interaction(s),"[{'interaction_number': 1, 'error_message': ' ...","[{'change_number': 1, 'before_code': ' if n..."
3,MAPLE_RECURSIVE_CONSEQ_2,sum,claude-sonnet-4-0,1,True,False,Verification passed after 1 LLM interaction(s),"[{'interaction_number': 1, 'error_message': ' ...","[{'change_number': 1, 'before_code': ' if n..."
4,MAPLE_RECURSIVE_CONSEQ_3,sum,claude-sonnet-4-0,1,True,False,Verification passed after 1 LLM interaction(s),"[{'interaction_number': 1, 'error_message': ' ...","[{'change_number': 1, 'before_code': ' if n..."


In [4]:
# Prepare DataFrames for joining - select relevant columns and rename llm_interactions
df_gpt4_join = df_gpt4[['class_name', 'feature_name', 'llm_interactions', 'success']].copy()
df_gpt4_join = df_gpt4_join.rename(columns={'llm_interactions': 'llm_interactions_gpt4', 'success': 'success_gpt4'})

df_sonnet_join = df_sonnet[['class_name', 'feature_name', 'llm_interactions', 'success']].copy()
df_sonnet_join = df_sonnet_join.rename(columns={'llm_interactions': 'llm_interactions_sonnet', 'success': 'success_sonnet'})

# Join on class_name and feature_name
df_joined = pd.merge(df_gpt4_join, df_sonnet_join, on=['class_name', 'feature_name'], how='inner')
print(f"Joined DataFrame shape: {df_joined.shape}")
df_joined.head()


Joined DataFrame shape: (26, 6)


,class_name,feature_name,llm_interactions_gpt4,success_gpt4,llm_interactions_sonnet,success_sonnet
0,MAPLE_RECURSIVE_ABSOLUTE_1,abs,1,True,1,True
1,MAPLE_RECURSIVE_ABSOLUTE_2,abs,1,True,1,True
2,MAPLE_RECURSIVE_CONSEQ_1,sum,2,True,1,True
3,MAPLE_RECURSIVE_CONSEQ_2,sum,6,True,1,True
4,MAPLE_RECURSIVE_CONSEQ_3,sum,6,True,1,True


In [5]:
# Calculate difference in interaction counts
df_joined['interaction_diff'] = df_joined['llm_interactions_gpt4'] - df_joined['llm_interactions_sonnet']

# Summary statistics
print("Summary Statistics for LLM Interactions:")
print("\nGPT-4:")
print(df_joined['llm_interactions_gpt4'].describe())
print("\nSonnet:")
print(df_joined['llm_interactions_sonnet'].describe())
print("\nDifference (GPT-4 - Sonnet):")
print(df_joined['interaction_diff'].describe())


Summary Statistics for LLM Interactions:

GPT-4:
count    26.000000
mean      2.576923
std       2.982255
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max      10.000000
Name: llm_interactions_gpt4, dtype: float64

Sonnet:
count    26.000000
mean      1.115385
std       0.325813
min       1.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       2.000000
Name: llm_interactions_sonnet, dtype: float64

Difference (GPT-4 - Sonnet):
count    26.000000
mean      1.461538
std       3.062427
min      -1.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       9.000000
Name: interaction_diff, dtype: float64


In [10]:
# Additional comparison metrics
print(f"\nTotal cases: {len(df_joined)}")
print(f"\nCases where GPT-4 required fewer interactions: {(df_joined['interaction_diff'] < 0).sum()}")
print(f"Cases where Sonnet required fewer interactions: {(df_joined['interaction_diff'] > 0).sum()}")
print(f"Cases with equal interactions: {(df_joined['interaction_diff'] == 0).sum()}")

print(f"\nMean interactions - GPT-4: {df_joined['llm_interactions_gpt4'].mean():.2f}")
print(f"Mean interactions - Sonnet: {df_joined['llm_interactions_sonnet'].mean():.2f}")
print(f"Mean difference: {df_joined['interaction_diff'].mean():.2f}")

# Display full comparison table
diff = df_joined[['class_name', 'feature_name', 'llm_interactions_gpt4', 'llm_interactions_sonnet', 'interaction_diff']].sort_values('interaction_diff')
diff[diff['interaction_diff'] != 0]



Total cases: 26

Cases where GPT-4 required fewer interactions: 3
Cases where Sonnet required fewer interactions: 8
Cases with equal interactions: 15

Mean interactions - GPT-4: 2.58
Mean interactions - Sonnet: 1.12
Mean difference: 1.46


,class_name,feature_name,llm_interactions_gpt4,llm_interactions_sonnet,interaction_diff
21,MAPLE_RECURSIVE_SUM_3_2,sum,1,2,-1
25,MAPLE_RECURSIVE_SUM_N_4,sum,1,2,-1
23,MAPLE_RECURSIVE_SUM_N_2,sum,1,2,-1
10,MAPLE_RECURSIVE_MAX_2_1,max,2,1,1
2,MAPLE_RECURSIVE_CONSEQ_1,sum,2,1,1
8,MAPLE_RECURSIVE_INCREMENT_3,increment,4,1,3
3,MAPLE_RECURSIVE_CONSEQ_2,sum,6,1,5
4,MAPLE_RECURSIVE_CONSEQ_3,sum,6,1,5
5,MAPLE_RECURSIVE_CONSEQ_4,sum,9,1,8
9,MAPLE_RECURSIVE_INCREMENT_4,increment,10,1,9
